# Chapter 15 - Naive Bayes Classifier

In this chapter, we study Naive Bayes - a classification algorithm based on
Bayes' theorem. 

The main idea is simple: given some evidence, which class is
most likely?

Naive Bayes is called "naive" because it assumes that input features are
conditionally independent once the class is known. This assumption is often
imperfect, but it makes the model fast, simple, and surprisingly useful,
especially for text classification and small datasets.

## Step 1 - Build the Play Tennis Example

In [2]:
# import the required libraries
import re

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB, GaussianNB, MultinomialNB

We will use a simple Play Tennis dataset where each row tells us
whether tennis was played under a particular weather condition.

In [4]:
play_tennis = pd.DataFrame(
    {
        "Day": range(1, 15),
        "Outlook": [
            "Sunny",
            "Sunny",
            "Overcast",
            "Rain",
            "Rain",
            "Rain",
            "Overcast",
            "Sunny",
            "Sunny",
            "Rain",
            "Sunny",
            "Overcast",
            "Overcast",
            "Rain",
        ],
        "Temp": [
            "Hot",
            "Hot",
            "Hot",
            "Mild",
            "Cool",
            "Cool",
            "Cool",
            "Mild",
            "Cool",
            "Mild",
            "Mild",
            "Mild",
            "Hot",
            "Mild",
        ],
        "Humidity": [
            "High",
            "High",
            "High",
            "High",
            "Normal",
            "Normal",
            "Normal",
            "High",
            "Normal",
            "Normal",
            "Normal",
            "High",
            "Normal",
            "High",
        ],
        "Wind": [
            "Weak",
            "Strong",
            "Weak",
            "Weak",
            "Weak",
            "Strong",
            "Strong",
            "Weak",
            "Weak",
            "Weak",
            "Strong",
            "Strong",
            "Weak",
            "Strong",
        ],
        "Play": [
            "No",
            "No",
            "Yes",
            "Yes",
            "Yes",
            "No",
            "Yes",
            "No",
            "Yes",
            "Yes",
            "Yes",
            "Yes",
            "Yes",
            "No",
        ],
    }
)

print("Play Tennis dataset:")
print(play_tennis)

Play Tennis dataset:
    Day   Outlook  Temp Humidity    Wind Play
0     1     Sunny   Hot     High    Weak   No
1     2     Sunny   Hot     High  Strong   No
2     3  Overcast   Hot     High    Weak  Yes
3     4      Rain  Mild     High    Weak  Yes
4     5      Rain  Cool   Normal    Weak  Yes
5     6      Rain  Cool   Normal  Strong   No
6     7  Overcast  Cool   Normal  Strong  Yes
7     8     Sunny  Mild     High    Weak   No
8     9     Sunny  Cool   Normal    Weak  Yes
9    10      Rain  Mild   Normal    Weak  Yes
10   11     Sunny  Mild   Normal  Strong  Yes
11   12  Overcast  Mild     High  Strong  Yes
12   13  Overcast   Hot   Normal    Weak  Yes
13   14      Rain  Mild     High  Strong   No


The dataset has four input features: `Outlook`, `Temp`, `Humidity`, and
`Wind`. 

The target column is `Play`, which contains the class label `Yes` or
`No`.

## Step 2 - Calculate Priors

The prior probability tells us how common each class is before looking at the
weather conditions.

In [5]:
class_counts = play_tennis["Play"].value_counts()
priors = class_counts / len(play_tennis)

print("Class counts:")
print(class_counts)

print("\nPrior probabilities:")
print(priors)

Class counts:
Play
Yes    9
No     5
Name: count, dtype: int64

Prior probabilities:
Play
Yes    0.642857
No     0.357143
Name: count, dtype: float64


The `Yes` class appears 9 times and the `No` class appears 5 times. 

So the model starts with `P(Yes) = 9/14` and `P(No) = 5/14`.

## Step 3 - Compare Class Scores With Naive Bayes

Now we answer the question
- Should we play tennis when the weather is `Sunny`, `Cool`, `High` humidity,
and `Strong` wind?

Naive Bayes multiplies the prior probability of a class by the conditional
probabilities of the feature values under that class.

In [6]:
query = {
    "Outlook": "Sunny",
    "Temp": "Cool",
    "Humidity": "High",
    "Wind": "Strong",
}


def conditional_probability(data, feature, value, target_class):
    """Return P(feature = value | Play = target_class)."""
    class_rows = data[data["Play"] == target_class]
    matching_rows = class_rows[class_rows[feature] == value]
    return len(matching_rows) / len(class_rows)


def naive_bayes_score(data, query_values, target_class):
    """Multiply the prior by each feature likelihood for one class."""
    score = priors[target_class]
    details = {}

    for feature, value in query_values.items():
        probability = conditional_probability(data, feature, value, target_class)
        details[f"P({value} | {target_class})"] = probability
        score *= probability

    return score, details


for target_class in ["Yes", "No"]:
    score, probability_details = naive_bayes_score(play_tennis, query, target_class)
    print(f"\nClass: {target_class}")
    print(pd.Series(probability_details))
    print(f"Final score: {score:.6f}")


Class: Yes
P(Sunny | Yes)     0.222222
P(Cool | Yes)      0.333333
P(High | Yes)      0.333333
P(Strong | Yes)    0.333333
dtype: float64
Final score: 0.005291

Class: No
P(Sunny | No)     0.6
P(Cool | No)      0.2
P(High | No)      0.8
P(Strong | No)    0.6
dtype: float64
Final score: 0.020571


We compare only the two scores. 

The denominator in Bayes' theorem is the same
for both classes, so it is not needed for deciding which class is larger.

In [9]:
scores = {
    target_class: float(naive_bayes_score(play_tennis, query, target_class)[0])
    for target_class in ["Yes", "No"]
}

prediction = max(scores, key=scores.get)

print("Scores:", scores)
print("\nPrediction for the query:", prediction)

Scores: {'Yes': 0.005291005291005291, 'No': 0.02057142857142857}

Prediction for the query: No


For this query, the `No` score is larger. 

So the model predicts that tennis will not be played under these weather conditions.

## Step 4 - Additive Smoothing

A zero probability can destroy a Naive Bayes score, because all probabilities
are multiplied together. 

Additive smoothing avoids this by adding a small
constant to the counts.

With Laplace smoothing, alpha is 1. 
- This means every possible feature value is treated as if it appeared once.

In [10]:
def smoothed_conditional_probability(data, feature, value, target_class, alpha=1):
    """Return a Laplace-smoothed P(feature = value | Play = target_class)."""
    class_rows = data[data["Play"] == target_class]
    observed_count = (class_rows[feature] == value).sum()

    # Include the queried value so an unseen value like "Foggy" still has a defined probability.
    possible_values = set(data[feature].unique()) | {value}
    number_of_values = len(possible_values)

    numerator = observed_count + alpha
    denominator = len(class_rows) + alpha * number_of_values
    return numerator / denominator


outlook_example = pd.DataFrame(
    {
        "Calculation": [
            "Without smoothing: P(Sunny | Yes)",
            "With Laplace smoothing: P(Sunny | Yes)",
            "With Laplace smoothing: P(Foggy | Yes)",
        ],
        "Probability": [
            conditional_probability(play_tennis, "Outlook", "Sunny", "Yes"),
            smoothed_conditional_probability(play_tennis, "Outlook", "Sunny", "Yes"),
            smoothed_conditional_probability(play_tennis, "Outlook", "Foggy", "Yes"),
        ],
    }
)

print(outlook_example)

                              Calculation  Probability
0       Without smoothing: P(Sunny | Yes)     0.222222
1  With Laplace smoothing: P(Sunny | Yes)     0.250000
2  With Laplace smoothing: P(Foggy | Yes)     0.076923


Even though "Foggy" does not exist anywhere in our input dataset.     
- So, by logic - it's probability should be zero
- But since we are using laplace smoothing here, it made sure that the final output never become zero when input like "Foggy" comes.

Smoothing slightly adjusts known probabilities and also gives unseen values a
small non-zero probability. 

This is especially important in text
classification, where many words may not appear in every class.

## Step 5 - Text Classification With Multinomial Naive Bayes

For text, each email can be represented by word counts. 

This makes `MultinomialNB` a natural choice because it is designed for count-based
features.

In [11]:
emails = pd.DataFrame(
    {
        "Text": [
            "Win a free iPhone now",
            "Meeting schedule tomorrow",
            "Lowest price guaranteed",
            "Limited time access premium features",
            "Amazon order shipped",
            "Congratulations won lottery",
            "Final reminder complete payment",
            "Team agenda attached",
            "Get cheap insurance quotes today",
            "Project deadline extended next week",
        ],
        "Label": [
            "Spam",
            "Not Spam",
            "Spam",
            "Spam",
            "Not Spam",
            "Spam",
            "Spam",
            "Not Spam",
            "Spam",
            "Not Spam",
        ],
    }
)

print("Training emails:")
print(emails)

Training emails:
                                   Text     Label
0                 Win a free iPhone now      Spam
1             Meeting schedule tomorrow  Not Spam
2               Lowest price guaranteed      Spam
3  Limited time access premium features      Spam
4                  Amazon order shipped  Not Spam
5           Congratulations won lottery      Spam
6       Final reminder complete payment      Spam
7                  Team agenda attached  Not Spam
8      Get cheap insurance quotes today      Spam
9   Project deadline extended next week  Not Spam


Before vectorization, the emails are plain text. 

The model cannot consume this
text directly, so we convert it into a table of word counts.

In [12]:
stop_words = ["a", "the", "and", "for"]
vectorizer = CountVectorizer(lowercase=True, stop_words=stop_words)
email_word_counts = vectorizer.fit_transform(emails["Text"])
vocabulary = vectorizer.get_feature_names_out()

word_count_table = pd.DataFrame(
    email_word_counts.toarray(),
    columns=vocabulary,
)

print("Vocabulary:")
print(vocabulary)

print("\nFirst five rows of word-count features:")
print(word_count_table.head())

Vocabulary:
['access' 'agenda' 'amazon' 'attached' 'cheap' 'complete'
 'congratulations' 'deadline' 'extended' 'features' 'final' 'free' 'get'
 'guaranteed' 'insurance' 'iphone' 'limited' 'lottery' 'lowest' 'meeting'
 'next' 'now' 'order' 'payment' 'premium' 'price' 'project' 'quotes'
 'reminder' 'schedule' 'shipped' 'team' 'time' 'today' 'tomorrow' 'week'
 'win' 'won']

First five rows of word-count features:
   access  agenda  amazon  attached  cheap  complete  congratulations  \
0       0       0       0         0      0         0                0   
1       0       0       0         0      0         0                0   
2       0       0       0         0      0         0                0   
3       1       0       0         0      0         0                0   
4       0       0       1         0      0         0                0   

   deadline  extended  features  ...  reminder  schedule  shipped  team  time  \
0         0         0         0  ...         0         0        0 

Each row is one email, each column is one word, and each value is the count of
that word in that email.

In [13]:
text_model = MultinomialNB()
text_model.fit(email_word_counts, emails["Label"])

new_email = ["Free quotes for premium insurance plan"]
new_email_counts = vectorizer.transform(new_email)

new_email_table = pd.DataFrame(
    new_email_counts.toarray(),
    columns=vocabulary,
)

non_zero_words = new_email_table.loc[:, new_email_table.iloc[0] > 0]

print("New email:")
print(new_email[0])

print("\nWords from the training vocabulary found in the new email:")
print(non_zero_words)

print("\nPredicted class:")
print(text_model.predict(new_email_counts)[0])

print("\nClass probabilities:")
print(
    pd.Series(
        text_model.predict_proba(new_email_counts)[0],
        index=text_model.classes_,
    )
)

New email:
Free quotes for premium insurance plan

Words from the training vocabulary found in the new email:
   free  insurance  premium  quotes
0     1          1        1       1

Predicted class:
Spam

Class probabilities:
Not Spam    0.077666
Spam        0.922334
dtype: float64


The word `plan` is ignored because it was not part of the training vocabulary.
The prediction is based only on the words that the model learned during
training.

## Step 6 - Numerical Data Uses Gaussian Naive Bayes

Counting works for categories and word frequencies. 

For continuous numerical features, exact matches are not useful because values may be unique. 

Gaussian Naive Bayes handles this by modeling each feature with a bell curve inside
each class.

Let's demonstrates this using the Iris dataset.

In [14]:
# load the iris dataset
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="species")

print("Iris feature preview:")
print(X.head())

print("\nTarget names:")
print({index: str(name) for index, name in enumerate(iris.target_names)})

Iris feature preview:
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
0                5.1               3.5                1.4               0.2
1                4.9               3.0                1.4               0.2
2                4.7               3.2                1.3               0.2
3                4.6               3.1                1.5               0.2
4                5.0               3.6                1.4               0.2

Target names:
{0: 'setosa', 1: 'versicolor', 2: 'virginica'}


The Iris features are continuous measurements, such as sepal length and petal
width. 

Because these are numerical measurements, `GaussianNB` is the correct
Naive Bayes variant.

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (120, 4)
Testing shape: (30, 4)


The split keeps some rows aside for testing. This lets us check how the model
behaves on data it did not see during training.

In [16]:
gaussian_model = GaussianNB()
gaussian_model.fit(X_train, y_train)

print("Class prior probabilities learned by GaussianNB:")
print(pd.Series(gaussian_model.class_prior_, index=iris.target_names))

print("\nFeature means learned within each class:")
print(pd.DataFrame(gaussian_model.theta_, index=iris.target_names, columns=X.columns))

Class prior probabilities learned by GaussianNB:
setosa        0.333333
versicolor    0.333333
virginica     0.333333
dtype: float64

Feature means learned within each class:
            sepal length (cm)  sepal width (cm)  petal length (cm)  \
setosa                  4.985             3.415             1.4775   
versicolor              5.930             2.750             4.2525   
virginica               6.610             2.980             5.5800   

            petal width (cm)  
setosa                 0.255  
versicolor             1.320  
virginica              2.040  


During fitting, `GaussianNB` estimates the mean and variance of each feature
inside each class. These are the bell-curve statistics which we discussed in the
chapter.

In [17]:
y_pred = gaussian_model.predict(X_test)

prediction_preview = X_test.copy()
prediction_preview["actual_species"] = [iris.target_names[i] for i in y_test]
prediction_preview["predicted_species"] = [iris.target_names[i] for i in y_pred]

print("Prediction preview:")
print(prediction_preview.head(10))

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Prediction preview:
     sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
38                 4.4               3.0                1.3               0.2   
127                6.1               3.0                4.9               1.8   
57                 4.9               2.4                3.3               1.0   
93                 5.0               2.3                3.3               1.0   
42                 4.4               3.2                1.3               0.2   
56                 6.3               3.3                4.7               1.6   
22                 4.6               3.6                1.0               0.2   
20                 5.4               3.4                1.7               0.2   
147                6.5               3.0                5.2               2.0   
84                 5.4               3.0                4.5               1.5   

    actual_species predicted_species  
38          setosa            setosa  
127      v

The above report shows the overall accuracy and the class-wise precision, recall,
and F1-score.

On a clean dataset like Iris, Gaussian Naive Bayes usually performs well even though the model is simple.

## Step 7 - Choose the Variant That Matches the Data

Scikit-learn provides different Naive Bayes variants. The important point is
to match the model to the feature type.

In [18]:
variant_guide = pd.DataFrame(
    {
        "Variant": ["GaussianNB", "MultinomialNB", "CategoricalNB"],
        "Best for": [
            "Continuous numerical features",
            "Counts or frequencies",
            "Discrete categorical labels",
        ],
        "Chapter example": [
            "Iris flower measurements",
            "Email word counts",
            "Encoded weather categories",
        ],
    }
)

print(variant_guide)

gaussian_example = GaussianNB()
multinomial_example = MultinomialNB()
categorical_example = CategoricalNB()

print("\nCreated model objects:")
print(type(gaussian_example).__name__)
print(type(multinomial_example).__name__)
print(type(categorical_example).__name__)

         Variant                       Best for             Chapter example
0     GaussianNB  Continuous numerical features    Iris flower measurements
1  MultinomialNB          Counts or frequencies           Email word counts
2  CategoricalNB    Discrete categorical labels  Encoded weather categories

Created model objects:
GaussianNB
MultinomialNB
CategoricalNB


`GaussianNB` is for continuous values, `MultinomialNB` is for counts, and
`CategoricalNB` is for encoded categories. 

Choosing the wrong variant can make
the model assumptions invalid.

## Step 8 - Advantages of Naive Bayes

Naive Bayes is simple, but it is very useful in practice.

- **Fast to train and predict:** it mainly calculates counts, means, variances, and probabilities.
- **Works well with high-dimensional data:** this is why it is often used for text classification, where each
  word can become a feature.
- **Useful on small datasets:** it can still learn reasonable patterns even when the training data is limited.
- **Easy to interpret:** we can inspect which words, feature values, or feature distributions push the model
  toward a class.
- **Strong baseline model:** it is often a good first model before trying more complex algorithms.

## Step 9 - Limitations of Naive Bayes

Naive Bayes also has important limitations:

- **Independence assumption is often unrealistic:** real features are usually related to each other.
- **Gaussian assumption may not always fit numerical data:** continuous features may be skewed, multimodal, or not bell-shaped.
- **Decision boundaries are simple:** the model may underfit when the true class separation is complex.
- **Mixed feature types are inconvenient:** numerical, categorical, and count features often need different Naive  Bayes variants.
- **Text results depend heavily on preprocessing:** tokenization, casing, stopword removal, and vocabulary choices can change performance.



## Chapter Recap

In this chapter we studied Naive Bayes.

- Naive Bayes uses Bayes' theorem to compare class probabilities.
- The "naive" assumption treats features as conditionally independent given the class.
- Additive smoothing prevents unseen feature values from making a full class score zero.
- Multinomial Naive Bayes works well for count-based data like word counts.
- Gaussian Naive Bayes works with continuous numerical features by modeling each feature with a bell curve.
- Categorical Naive Bayes is used when features are discrete categories.
- Naive Bayes is fast, interpretable, and useful as a baseline, but its assumptions can limit accuracy on more
complex problems.